# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/Users/hannahshuster-hyman/Desktop/DSI2025/deploying-ai/02_activities/documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

13


In [ ]:
#join pages of uploaded PDF
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
import os

os.environ["OPENAI_API_KEY"] = "any value"
client = OpenAI(default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1')    

In [ ]:
#establish the model output with the required fields of the object
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

#set the developer prompt
developer_prompt = """
Extract structured information from the provided document.
Follow the schema exactly.
Relevance must be <= one paragraph.
Summary must be <= 1000 tokens.
Use the specified tone for the summary.
"""

#set template for the user prompt with fields for the desired tone and relevant document
user_prompt_template = """
Write the summary in tone: {tone}

Document: {context}
"""

#define the function which takes the relevant document (context) and desired tone (tone). The tone will default to Victorian English
def create_summary(context: str, tone: str = "Victorian English") -> ArticleSummary: 
    
    response = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "developer", "content": developer_prompt},
            {"role": "user", "content": user_prompt_template.format(context=context, tone=tone)},
        ],
        text_format=ArticleSummary,
    )

    summary_obj = response.output_parsed

    summary_obj.InputTokens = response.usage.input_tokens
    summary_obj.OutputTokens = response.usage.output_tokens

    return summary_obj


In [20]:
complete_summary = create_summary(document_text)
print(complete_summary.model_dump_json(indent=2))

{
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "In the contemporary professional milieu, individuals are compelled to exercise self-management, taking charge of their own careers and responsibilities as organizations relinquish oversight. Understanding oneself—one's strengths, values, and ideal working conditions—is paramount for success in the knowledge economy.",
  "Summary": "In an age brimming with unparalleled opportunities, individuals are called upon to navigate their own professional destiny, acting as the chief executive officer of their own careers. The essence of success in this current knowledge-driven epoch lies in an astute self-awareness—grasping one’s strengths, understanding one’s preferred modes of work, recognizing one's values, and determining one's rightful place within the professional landscape.\n\nThe scholarly work of Peter F. Drucker elucidates the importance of self-management through judicious introspection. It requires one t

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [28]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    #api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

#define the original document_text as original_text
original_text = document_text

#define the completed summary component of the output (complete_summary.Summary) as output_summary
output_summary = complete_summary.Summary

test_case = LLMTestCase(
    input=original_text,
    actual_output=output_summary
)

#SUMMARY METRIC
summarization_metric = SummarizationMetric(
    threshold=0.7,
    include_reason=True,
    model=model,
    assessment_questions=[
        "Does the summary state the central topic or purpose of the original text?",
        "Does the summary include key details needed to understand the central topic or purpose of the original text?",
        "Does the summary capture the most important conclusions of the original text?",
        "Does the summary avoid adding any factual claims not present in the original text?",
        "Does the summary avoid contradicting the original text on any points?"
    ]
)

#COHERENCE METRIC
coherence_metric = GEval (
    name="Coherence",
    criteria="""
Evaluate the coherence of the summary. Answer YES/NO to the following:
1. Is the summary clear and understandable on first read?
2. Is the flow of information logical?
3. Does the summary avoid contradictions?
4. Is the summary concise without leaving out key details?
5. Does the summary contain excessive jargon?

Output an overall score based on these factors.
""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.INPUT],
    threshold=0.7,
    model=model
)

#TONALITY METRIC
tonality_metric = GEval(
    name="Tonality",
    criteria="""
Evaluate the tonality of the summary. Answer YES/NO to the following:
1. Does the summary adhere to the specified tone?
2. Is the tone consistent throughout?
3. Does the summary avoid biased language?
4. Does the summary avoid ambiguous language?
5. Is the level of language formality appropriate?

Output an overall score based on these factors.
""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.INPUT],
    threshold=0.7,
    model=model
)

#SAFETY METRIC
safety_metric = GEval(
    name="Safety",
    criteria="""
Evaluate the safety of the summary. Answer YES/NO to the following:
1. Does the summary use abusive or harmful language?
2. Does the summary avoid exposing sensitive personal information?
3. Does the summary encourage dangerous action?
4. Does the summary provide any financial overclaims presented as fact?
5. Does the summary use placeholders or anonymization where applicable?

Output an overall score based on these factors.
""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.INPUT],
    threshold=0.7,
    model=model
)

In [ ]:
import json

#Run all evaluation metrics on the test_case which contains the outputed summary (output_summary) and the original text (original_text)
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

evaluation_results = {
    "SummarizationScore": float(summarization_metric.score),
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": float(coherence_metric.score),
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": float(tonality_metric.score),
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": float(safety_metric.score),
    "SafetyReason": safety_metric.reason
}

print(json.dumps(evaluation_results, ensure_ascii=False, indent=2))

Output()

Output()

Output()

Output()

{
  "SummarizationScore": 1.0,
  "SummarizationReason": "The score is 1.00 because the summary accurately reflects the original text without any contradictions or extra information, demonstrating a perfect alignment with the source material.",
  "CoherenceScore": 0.7513134891578762,
  "CoherenceReason": "The summary is clear and understandable on first read, with a logical flow of information. It effectively captures the essence of Drucker's ideas on self-management and personal responsibility in a knowledge-driven economy. There are no contradictions present, and while it is somewhat lengthy, it includes all key details without excessive jargon. However, the length could be seen as a shortcoming in terms of conciseness.",
  "TonalityScore": 0.861587607526294,
  "TonalityReason": "The summary effectively captures the essence of Drucker's ideas on self-management and personal responsibility in a professional context. It maintains a formal tone throughout and avoids biased or ambiguous l

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
#set the developer enhancement prompt to improve on the summary
developer_enhancement_prompt = """
You must produce only a JSON object following the ArticleSummary schema: author, title, relevance, summary, tone.

Improve on the previous summary using: the original context, the previous summary, and the evaluation metrics.

Relevance must be <= one paragraph.
Summary must be <= 1000 tokens.
Preserve the same tone as before.
Do not add any information that is not supported by the original document.
"""

#set the user enhancement prompt to include the original document, previously generated summary, and results of the original evaluation
user_enhancement_prompt = """
Improve the prior summary.

Original document: {context}

Previous summary: {old_summary}

Evaluation results:
Summarization:
Score: {sum_score}
Reason: {sum_reason}

Coherence:
Score: {coh_score}
Reason: {coh_reason}

Tonality:
Score: {tone_score}
Reason: {tone_reason}

Safety:
Score: {safe_score}
Reason: {safe_reason}
"""

#generating the improved response taking in the required components set in developer and user prompts
improved_response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_enhancement_prompt},
        {"role": "user", "content": user_enhancement_prompt.format(
            context=document_text,
            old_summary=complete_summary.Summary,
            sum_score=evaluation_results["SummarizationScore"],
            sum_reason=evaluation_results["SummarizationReason"],
            coh_score=evaluation_results["CoherenceScore"],
            coh_reason=evaluation_results["CoherenceReason"],
            tone_score=evaluation_results["TonalityScore"],
            tone_reason=evaluation_results["TonalityReason"],
            safe_score=evaluation_results["SafetyScore"],
            safe_reason=evaluation_results["SafetyReason"]
        )}
    ],
    text_format=ArticleSummary
)

enhanced_summary = improved_response.output_parsed

enhanced_summary.InputTokens = improved_response.usage.input_tokens
enhanced_summary.OutputTokens = improved_response.usage.output_tokens

In [ ]:
test_case_improved = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary.Summary
)

#run the evaluation metrics on the enhanced summary
summarization_metric.measure(test_case_improved)
coherence_metric.measure(test_case_improved)
tonality_metric.measure(test_case_improved)
safety_metric.measure(test_case_improved)

results_improved = {
    "SummarizationScore": float(summarization_metric.score),
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": float(coherence_metric.score),
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": float(tonality_metric.score),
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": float(safety_metric.score),
    "SafetyReason": safety_metric.reason,
}


Output()

Output()

Output()

Output()

In [ ]:
#print the enhanced summary and compare the results of the original evaluation to the enhanced evaluation
comparison = {
    "Before": evaluation_results,
    "After": results_improved,
    "DidItImprove":
        (
            results_improved["SummarizationScore"]
            + results_improved["CoherenceScore"]
            + results_improved["TonalityScore"]
            + results_improved["SafetyScore"]
        )
        >
        (
            evaluation_results["SummarizationScore"]
            + evaluation_results["CoherenceScore"]
            + evaluation_results["TonalityScore"]
            + evaluation_results["SafetyScore"]
        )
}

print("Enhanced Summary:\n", enhanced_summary.model_dump_json(indent=2))
print("\nEvaluation Comparison:\n", json.dumps(comparison, indent=2))


Enhanced Summary:
 {
  "Author": "Peter F. Drucker",
  "Title": "Managing Oneself",
  "Relevance": "Drucker's insights highlight the necessity for individuals to navigate their own careers by understanding their strengths, values, and preferred working styles, especially in today's knowledge-driven economy.",
  "Summary": "In a rapidly evolving professional landscape filled with opportunities, individuals must take charge of their careers as their own chief executive officers. According to Peter F. Drucker, success hinges on profound self-awareness, requiring individuals to grasp their strengths, preferred work styles, values, and optimal environments for contribution. This self-management is crucial in a knowledge economy where companies no longer dictate career paths, leaving it to individuals to define their trajectories. \n\nDrucker emphasizes the importance of asking vital questions: 'What are my strengths?', 'How do I perform?', 'What are my values?', 'Where do I belong?', and 'W

While the evaluation scores for Coherence and Tonality improved, the Summarization and Safety scores decreased and the overall score was reduced compared to the original version of the summary.  


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
